In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, MinMaxScaler

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
spark = SparkSession.builder \
    .appName("F1_MinMaxScaler_Demo") \
    .getOrCreate()

print("Spark session successfully created.")

26/06/11 10:11:32 WARN Utils: Your hostname, Abhisheks-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.22.74.127 instead (on interface en0)
26/06/11 10:11:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 10:11:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session successfully created.


In [3]:
df = spark.read.csv("f1.csv", header=True, inferSchema=True)

In [4]:
print("Original F1 Data:")
df.show()
df.printSchema()

Original F1 Data:
+---------------+---------------+-------------+
|    driver_name|           team|top_speed_kmh|
+---------------+---------------+-------------+
| Max Verstappen|Red Bull Racing|        345.5|
|   Lando Norris|        McLaren|        343.2|
|Charles Leclerc|        Ferrari|        344.8|
| Lewis Hamilton|       Mercedes|        341.0|
|Fernando Alonso|   Aston Martin|        338.5|
|   Pierre Gasly|         Alpine|        335.0|
|Alexander Albon|       Williams|        342.1|
|   Yuki Tsunoda|             RB|        337.4|
|Valtteri Bottas|    Kick Sauber|        334.2|
|Nico Hulkenberg|           Haas|        339.8|
+---------------+---------------+-------------+

root
 |-- driver_name: string (nullable = true)
 |-- team: string (nullable = true)
 |-- top_speed_kmh: double (nullable = true)



In [5]:
assembler = VectorAssembler(
    inputCols=["top_speed_kmh"],
    outputCol="features"
)

feature_df = assembler.transform(df)

print("Data after VectorAssembler:")
feature_df.select("driver_name", "top_speed_kmh", "features").show()

Data after VectorAssembler:
+---------------+-------------+--------+
|    driver_name|top_speed_kmh|features|
+---------------+-------------+--------+
| Max Verstappen|        345.5| [345.5]|
|   Lando Norris|        343.2| [343.2]|
|Charles Leclerc|        344.8| [344.8]|
| Lewis Hamilton|        341.0| [341.0]|
|Fernando Alonso|        338.5| [338.5]|
|   Pierre Gasly|        335.0| [335.0]|
|Alexander Albon|        342.1| [342.1]|
|   Yuki Tsunoda|        337.4| [337.4]|
|Valtteri Bottas|        334.2| [334.2]|
|Nico Hulkenberg|        339.8| [339.8]|
+---------------+-------------+--------+



In [6]:
scaler = MinMaxScaler(
    inputCol="features",
    outputCol="scaled_features"
)

In [7]:
scaler_model = scaler.fit(feature_df)
print(f"Min: {scaler_model.originalMin}, Max: {scaler_model.originalMax}")

Min: [334.2], Max: [345.5]


In [8]:
scaled_df = scaler_model.transform(feature_df)

In [9]:
print("Final Scaled F1 Data:")
scaled_df.select("driver_name", "team", "top_speed_kmh", "scaled_features").show(truncate=False)

Final Scaled F1 Data:
+---------------+---------------+-------------+--------------------+
|driver_name    |team           |top_speed_kmh|scaled_features     |
+---------------+---------------+-------------+--------------------+
|Max Verstappen |Red Bull Racing|345.5        |[1.0]               |
|Lando Norris   |McLaren        |343.2        |[0.7964601769911497]|
|Charles Leclerc|Ferrari        |344.8        |[0.9380530973451339]|
|Lewis Hamilton |Mercedes       |341.0        |[0.6017699115044252]|
|Fernando Alonso|Aston Martin   |338.5        |[0.3805309734513281]|
|Pierre Gasly   |Alpine         |335.0        |[0.0707964601769921]|
|Alexander Albon|Williams       |342.1        |[0.6991150442477899]|
|Yuki Tsunoda   |RB             |337.4        |[0.2831858407079633]|
|Valtteri Bottas|Kick Sauber    |334.2        |[0.0]               |
|Nico Hulkenberg|Haas           |339.8        |[0.4955752212389396]|
+---------------+---------------+-------------+--------------------+

